# Cluster + Claude Code Test
Run on GPU node to verify Claude Code works there.

```bash
srun --partition=dev_gpu_h100 --time=00:10:00 --gres=gpu:1 --mem=64G \
  uv run jupyter nbconvert --to notebook --execute --inplace notebooks/test_cluster_claude.ipynb
```

If all tests pass → use Mode A (sbatch). If test 2 fails → use Mode B (tmux).

## Test 1: GPU visible

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(result.stdout.strip())
assert result.returncode == 0, 'nvidia-smi failed — no GPU available'
print('PASS: GPU visible')

## Test 2: Claude Code works

In [ ]:
import subprocess

# Check claude is installed
result = subprocess.run(['claude', '--version'], capture_output=True, text=True)
print(f'claude version: {result.stdout.strip()}')
assert result.returncode == 0, f'claude --version failed: {result.stderr}'

# Check claude can respond
result = subprocess.run(
    ['claude', '-p', 'respond with exactly: HELLO', '--dangerously-skip-permissions', '--model', 'sonnet'],
    capture_output=True, text=True, timeout=60,
)
print(f'claude response: {result.stdout.strip()[:100]}')
assert result.returncode == 0, f'claude -p failed: {result.stderr}'
assert 'HELLO' in result.stdout.upper(), f'unexpected response: {result.stdout[:200]}'
print('PASS: Claude Code works on this node')

## Test 3: main.py runs (quick 5-step test)

In [ ]:
import subprocess, os

repo_root = os.path.join(os.path.dirname(os.path.abspath('')), '')
result = subprocess.run(
    ['uv', 'run', 'src/main.py', '--game', 'tower_of_hanoi', '--num_disks', '3', '--max_steps', '5'],
    capture_output=True, text=True, timeout=300, cwd=repo_root,
)
print('stdout (last 5 lines):')
for line in result.stdout.strip().split('\n')[-5:]:
    print(f'  {line}')
if result.returncode != 0:
    print(f'stderr: {result.stderr[-500:]}')
# Don't assert returncode — wandb/weave may cause non-zero exit but main.py still runs
assert 'Success Rate' in result.stdout, f'No Success Rate in output'
print('PASS: main.py runs on GPU')

## Summary

| Test | What | Result |
|------|------|--------|
| 1 | GPU visible (nvidia-smi) | See above |
| 2 | Claude Code responds | See above |
| 3 | main.py runs | See above |

All pass → **use Mode A** (sbatch with `program.md`)

Test 2 fails → **use Mode B** (tmux with `program_fallback.md`)